
# Algoritmo de Múltiplos Clusters de Swendsen-Wang para o Modelo de Ising 2D

**Material de apoio ao TCC:** *Comparação e Otimização de Algoritmos de Monte Carlo
Aplicados ao Modelo de Ising* (Diego R. Oliveira, UFPR).

Este notebook implementa o algoritmo de **Swendsen-Wang**, seguindo
fielmente a formulação da Seção **4.3.3 (Algoritmo de Múltiplos Clusters de
Swendsen-Wang)** do TCC.

**Estrutura deste notebook:**
1. Construção das ligações (*bonds*)
2. Identificação dos aglomerados (equivalente a Hoshen-Kopelman via *Union-Find*)
3. Atualização coletiva e inversão
4. Um passo completo do algoritmo de Swendsen-Wang
5. Protocolo completo de simulação
6. Testes de sanidade internos
7. Validação preliminar contra a solução analítica de Onsager

**Nota de implementação importante:** a Seção 4.3.3.2 do TCC descreve o
algoritmo de **Hoshen-Kopelman** como método de identificação dos
aglomerados. Aqui implementamos essa mesma etapa usando a estrutura de dados
**Union-Find (conjuntos disjuntos) com compressão de caminho** -- uma forma
padrão e amplamente usada de resolver exatamente o mesmo problema
(rotulação de componentes conexas), com o mesmo resultado final. A escolha
é por clareza de implementação em Python; o resultado (quais sítios formam
qual aglomerado) é idêntico ao que o Hoshen-Kopelman produziria.

As funções do *sistema físico* (rede, vizinhos, energia, magnetização,
probabilidade de ligação, Onsager) são importadas de
[`ising_utils.py`](https://github.com/diegorafael1010/ising-monte-carlo/blob/main/ising_utils.py).


In [ ]:

!git clone https://github.com/diegorafael1010/ising-monte-carlo.git
import sys
sys.path.append('/content/ising-monte-carlo')


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from ising_utils import (
    inicializar_rede,
    construir_tabela_vizinhos,
    energia_total,
    magnetizacao_total,
    probabilidade_ligacao,
    temperatura_critica_onsager,
    energia_onsager,
    magnetizacao_onsager,
    J,
)

SEED = 42
rng = np.random.default_rng(SEED)



## 1. Construção das Ligações (*Bonds*)

Seguindo a Seção 4.3.3.1 do TCC, cada ligação entre primeiros vizinhos é
ativada, de forma independente, com probabilidade $P_{\text{add}} =
1-e^{-2\beta J}$ (Equação 4.17) **somente se os dois spins forem paralelos**;
se forem antiparalelos, a ligação nunca é ativada.

Diferente do Wolff (que cresce um único aglomerado a partir de uma semente),
aqui **todas** as ligações da rede são visitadas em cada passo. Para não
testar cada ligação duas vezes (a ligação entre os sítios $i$ e $j$ é a
mesma, seja vista "a partir de $i$" ou "a partir de $j$"), percorremos
apenas duas das quatro direções por sítio -- **sul** e **leste** -- o que,
somado sobre todos os sítios, cobre exatamente cada ligação da rede uma
única vez (a ligação norte de um sítio é a ligação sul do seu vizinho ao
norte, e assim por diante).


In [ ]:

# Índices das colunas na tabela de vizinhos (ver ising_utils.construir_tabela_vizinhos):
# vizinhos[:, 0] = norte, vizinhos[:, 1] = sul, vizinhos[:, 2] = leste, vizinhos[:, 3] = oeste
SUL, LESTE = 1, 2

def construir_ligacoes_ativas(
    estado: np.ndarray,
    vizinhos: np.ndarray,
    beta: float,
    rng: np.random.Generator,
) -> list:
    '''Percorre todas as ligações da rede (cada uma exatamente uma vez) e
    decide quais ficam ativas (Equação 4.17 do TCC).

    Returns
    -------
    list of tuple(int, int)
        Lista de pares (sitio_i, sitio_j) para cada ligação ativada.
    '''
    p_add = probabilidade_ligacao(beta)
    ligacoes_ativas = []

    for direcao in (SUL, LESTE):
        vizinho_na_direcao = vizinhos[:, direcao]
        mesmo_spin = estado == estado[vizinho_na_direcao]
        sorteio = rng.random(estado.shape[0]) < p_add
        ativas = np.nonzero(mesmo_spin & sorteio)[0]
        for i in ativas:
            ligacoes_ativas.append((int(i), int(vizinho_na_direcao[i])))

    return ligacoes_ativas


In [ ]:

# Teste de sanidade: em T muito baixa (p_add ~ 1), quase todas as ligações
# entre spins paralelos devem estar ativas. Numa rede fria (todos paralelos),
# isso significa quase todas as 2N ligações da rede.
L_teste = 16
estado_frio = inicializar_rede(L_teste, modo="fria")
viz_teste = construir_tabela_vizinhos(L_teste)
N_teste = L_teste ** 2

ligacoes_T_baixa = construir_ligacoes_ativas(estado_frio, viz_teste, beta=5.0, rng=rng)
print(f"Ligações ativas em T baixa: {len(ligacoes_T_baixa)} de {2*N_teste} possíveis")
assert len(ligacoes_T_baixa) > 0.95 * (2 * N_teste)

# Em T muito alta (p_add ~ 0), quase nenhuma ligação deve estar ativa.
ligacoes_T_alta = construir_ligacoes_ativas(estado_frio, viz_teste, beta=0.01, rng=rng)
print(f"Ligações ativas em T alta: {len(ligacoes_T_alta)} de {2*N_teste} possíveis")
assert len(ligacoes_T_alta) < 0.05 * (2 * N_teste)
print("OK: construir_ligacoes_ativas se comporta como esperado nos limites de T.")



## 2. Identificação dos Aglomerados (Union-Find)

Com as ligações ativas em mãos, é preciso agrupar os sítios em aglomerados
(componentes conexas). Implementamos isso com a estrutura clássica
**Union-Find** (também chamada *Disjoint Set Union*), usando duas otimizações
padrão que garantem eficiência quase linear:

- **Compressão de caminho** (*path compression*): ao buscar a raiz de um
  sítio, todos os sítios visitados no caminho passam a apontar diretamente
  para a raiz, acelerando buscas futuras.
- **União por tamanho** (*union by size*): ao unir dois aglomerados, a raiz
  do menor passa a apontar para a raiz do maior, mantendo as árvores rasas.

Isso corresponde, na prática, ao mesmo resultado do algoritmo de
Hoshen-Kopelman descrito na Seção 4.3.3.2 do TCC: ao final, cada sítio
recebe um rótulo (a raiz do seu aglomerado), e sítios com o mesmo rótulo
pertencem ao mesmo aglomerado.


In [ ]:

class UnionFind:
    '''Estrutura de conjuntos disjuntos (Union-Find) com compressão de
    caminho e união por tamanho, usada para identificar os aglomerados de
    percolação formados pelas ligações ativas (Seção 4.3.3.2 do TCC).
    '''

    def __init__(self, n: int):
        self.pai = np.arange(n)
        self.tamanho = np.ones(n, dtype=np.int64)

    def encontrar(self, i: int) -> int:
        '''Encontra a raiz do aglomerado de i, comprimindo o caminho.'''
        raiz = i
        while self.pai[raiz] != raiz:
            raiz = self.pai[raiz]
        # Compressão de caminho: todos os nós no caminho apontam direto para a raiz
        while self.pai[i] != raiz:
            self.pai[i], i = raiz, self.pai[i]
        return raiz

    def unir(self, i: int, j: int) -> None:
        '''Une os aglomerados de i e j (união por tamanho).'''
        raiz_i, raiz_j = self.encontrar(i), self.encontrar(j)
        if raiz_i == raiz_j:
            return
        if self.tamanho[raiz_i] < self.tamanho[raiz_j]:
            raiz_i, raiz_j = raiz_j, raiz_i
        self.pai[raiz_j] = raiz_i
        self.tamanho[raiz_i] += self.tamanho[raiz_j]

    def rotulos(self) -> np.ndarray:
        '''Retorna um array de tamanho N com o rótulo (raiz) do aglomerado
        de cada sítio. Sítios com o mesmo rótulo pertencem ao mesmo aglomerado.
        '''
        return np.array([self.encontrar(i) for i in range(len(self.pai))])


def identificar_aglomerados(N: int, ligacoes_ativas: list) -> np.ndarray:
    '''Identifica os aglomerados formados pelas ligações ativas
    (equivalente ao Hoshen-Kopelman da Seção 4.3.3.2 do TCC).

    Returns
    -------
    np.ndarray
        Array de tamanho N com o rótulo do aglomerado de cada sítio.
    '''
    uf = UnionFind(N)
    for i, j in ligacoes_ativas:
        uf.unir(i, j)
    return uf.rotulos()


In [ ]:

# Teste de sanidade: uma cadeia simples de ligações (0-1, 1-2, 2-3) deve unir
# os 4 sítios num único aglomerado, mesmo sem ligação direta entre 0 e 3.
uf_teste = UnionFind(6)
uf_teste.unir(0, 1)
uf_teste.unir(1, 2)
uf_teste.unir(2, 3)
rotulos_teste = uf_teste.rotulos()

assert rotulos_teste[0] == rotulos_teste[1] == rotulos_teste[2] == rotulos_teste[3]
assert rotulos_teste[4] != rotulos_teste[0]  # sítio isolado, aglomerado próprio
assert rotulos_teste[5] != rotulos_teste[0]  # outro sítio isolado
print("OK: Union-Find agrupa corretamente aglomerados encadeados e mantém sítios isolados separados.")



## 3. Atualização Coletiva e Inversão

Para cada aglomerado identificado, sorteia-se de forma independente uma
nova orientação, $+1$ ou $-1$, com igual probabilidade (Equação 4.18 do
TCC). Todos os sítios daquele aglomerado assumem essa mesma orientação
(Equação 4.19) -- **não** é uma inversão determinística como no Wolff, e
sim um novo sorteio: por isso, cerca de metade dos aglomerados "não muda"
de fato (sorteiam a mesma orientação que já tinham).


In [ ]:

def atualizar_aglomerados(estado: np.ndarray, rotulos: np.ndarray, rng: np.random.Generator) -> None:
    '''Sorteia uma nova orientação para cada aglomerado e a aplica a todos
    os seus sítios, in-place (Equações 4.18 e 4.19 do TCC).
    '''
    aglomerados_unicos = np.unique(rotulos)
    novos_spins = rng.choice(np.array([-1, 1], dtype=np.int8), size=aglomerados_unicos.shape[0])

    mapa_rotulo_para_spin = dict(zip(aglomerados_unicos.tolist(), novos_spins.tolist()))
    for i in range(estado.shape[0]):
        estado[i] = mapa_rotulo_para_spin[rotulos[i]]


In [ ]:

# Teste de sanidade: após a atualização, todo sítio com o mesmo rótulo deve
# ter exatamente o mesmo spin.
L_teste = 10
estado_teste = inicializar_rede(L_teste, modo="quente", rng=rng)
viz_teste = construir_tabela_vizinhos(L_teste)

ligacoes = construir_ligacoes_ativas(estado_teste, viz_teste, beta=0.5, rng=rng)
rotulos = identificar_aglomerados(L_teste ** 2, ligacoes)
atualizar_aglomerados(estado_teste, rotulos, rng)

for rotulo in np.unique(rotulos):
    spins_do_aglomerado = estado_teste[rotulos == rotulo]
    assert len(set(spins_do_aglomerado.tolist())) == 1, "Aglomerado com spins divergentes após atualização!"
print("OK: todos os sítios de cada aglomerado compartilham o mesmo spin após a atualização.")



## 4. Um Passo Completo do Algoritmo de Swendsen-Wang

Como discutido na Seção 4.3.3.4 do TCC, uma atualização do Swendsen-Wang
consiste na varredura, partição e reatualização de **toda a rede
simultaneamente** -- por isso, diferente do Wolff, **1 passo de
Swendsen-Wang equivale a 1 MCS/site**, a mesma escala temporal do
Metropolis.


In [ ]:

def passo_swendsen_wang(estado: np.ndarray, vizinhos: np.ndarray, beta: float, rng: np.random.Generator) -> int:
    '''Executa uma atualização completa do algoritmo de Swendsen-Wang:
    constrói as ligações, identifica os aglomerados e os reatualiza
    coletivamente (Seção 4.3.3 do TCC).

    Returns
    -------
    int
        Número de aglomerados distintos identificados nesse passo (útil
        como diagnóstico da fragmentação da rede).
    '''
    N = estado.shape[0]
    ligacoes = construir_ligacoes_ativas(estado, vizinhos, beta, rng)
    rotulos = identificar_aglomerados(N, ligacoes)
    atualizar_aglomerados(estado, rotulos, rng)
    return int(np.unique(rotulos).shape[0])


In [ ]:

# Teste rápido: em T alta, esperamos MUITOS aglomerados pequenos (rede
# fragmentada); em T baixa, esperamos POUCOS aglomerados grandes.
L_teste = 32
N_teste = L_teste ** 2
viz_teste = construir_tabela_vizinhos(L_teste)

estado_T_alta = inicializar_rede(L_teste, modo="quente", rng=rng)
n_aglomerados_T_alta = passo_swendsen_wang(estado_T_alta, viz_teste, beta=0.05, rng=rng)

estado_T_baixa = inicializar_rede(L_teste, modo="fria", rng=rng)
n_aglomerados_T_baixa = passo_swendsen_wang(estado_T_baixa, viz_teste, beta=1.5, rng=rng)

print(f"Número de aglomerados em T alta (beta=0.05): {n_aglomerados_T_alta} (de N={N_teste} sítios)")
print(f"Número de aglomerados em T baixa (beta=1.5): {n_aglomerados_T_baixa}")
assert n_aglomerados_T_alta > n_aglomerados_T_baixa
print("OK: a rede fragmenta em T alta e se funde em poucos aglomerados em T baixa, como esperado.")



## 5. Protocolo Completo de Simulação (Equilibração + Produção)

Segue o mesmo protocolo da Seção 4.5.3 do TCC, agora em unidades de passos
de Swendsen-Wang (que, como discutido acima, equivalem a MCS/site).


In [ ]:

def simular_swendsen_wang(
    L: int,
    T: float,
    n_eq: int,
    n_prod: int,
    intervalo_amostragem: int = 1,
    modo_inicial: str = "quente",
    rng: np.random.Generator = rng,
) -> dict:
    '''Executa o protocolo completo de simulação de Swendsen-Wang para o
    modelo de Ising 2D (Seções 4.3.3 e 4.5.3 do TCC).

    Returns
    -------
    dict
        Dicionário com "energia_por_sitio", "magnetizacao_abs_por_sitio",
        "numero_aglomerados" e os parâmetros usados (L, T, n_eq, n_prod).
    '''
    N = L * L
    beta = 1.0 / T

    estado = inicializar_rede(L, modo=modo_inicial, rng=rng)
    vizinhos = construir_tabela_vizinhos(L)

    for _ in range(n_eq):
        passo_swendsen_wang(estado, vizinhos, beta, rng)

    energias = []
    magnetizacoes = []
    numeros_aglomerados = []
    for passo in range(n_prod):
        n_aglomerados = passo_swendsen_wang(estado, vizinhos, beta, rng)
        if passo % intervalo_amostragem == 0:
            e = energia_total(estado, vizinhos) / N
            m = abs(magnetizacao_total(estado)) / N
            energias.append(e)
            magnetizacoes.append(m)
            numeros_aglomerados.append(n_aglomerados)

    return {
        "energia_por_sitio": np.array(energias),
        "magnetizacao_abs_por_sitio": np.array(magnetizacoes),
        "numero_aglomerados": np.array(numeros_aglomerados),
        "L": L,
        "T": T,
        "n_eq": n_eq,
        "n_prod": n_prod,
    }


In [ ]:

# Teste rápido (parâmetros pequenos só para conferir que a função roda --
# NÃO é uma rodada de produção real).
resultado_teste = simular_swendsen_wang(L=32, T=2.269, n_eq=50, n_prod=300)
print("Energia média por sítio:", resultado_teste["energia_por_sitio"].mean())
print("Magnetização absoluta média por sítio:", resultado_teste["magnetizacao_abs_por_sitio"].mean())
print("Número médio de aglomerados:", resultado_teste["numero_aglomerados"].mean())

plt.figure(figsize=(7, 3))
plt.plot(resultado_teste["energia_por_sitio"])
plt.xlabel("Amostra (1 por passo de Swendsen-Wang)")
plt.ylabel(r"Energia por sítio $e$")
plt.title(f"Traço de energia -- L={resultado_teste['L']}, T={resultado_teste['T']}")
plt.tight_layout()
plt.show()



## 6. Testes de Sanidade Internos

Um teste adicional relevante aqui: confirmar que a energia calculada logo
após a atualização coletiva é consistente com o cálculo direto a partir da
configuração final -- ou seja, que não há nenhum efeito colateral estranho
na forma como `atualizar_aglomerados` modifica o array `estado`.


In [ ]:

def teste_consistencia_energia_pos_atualizacao(L: int = 16, n_testes: int = 20, rng: np.random.Generator = rng) -> None:
    '''Confere que a energia calculada apos um passo de Swendsen-Wang bate
    com o calculo direto sobre a configuracao final resultante.
    '''
    vizinhos = construir_tabela_vizinhos(L)

    for _ in range(n_testes):
        estado = inicializar_rede(L, modo="quente", rng=rng)
        beta_aleatorio = rng.uniform(0.1, 1.0)

        passo_swendsen_wang(estado, vizinhos, beta_aleatorio, rng)

        # Recalcula a energia do zero, de forma totalmente independente da
        # simulação, e compara com o valor que energia_total() retornaria.
        soma_manual = 0.0
        for i in range(L * L):
            for viz in vizinhos[i]:
                soma_manual += -J * estado[i] * estado[viz]
        soma_manual /= 2.0  # cada ligação contada duas vezes

        assert np.isclose(soma_manual, energia_total(estado, vizinhos)), (
            "Energia recalculada manualmente diverge de energia_total()!"
        )

    print(f"OK: energia consistente em {n_testes} configurações pós-atualização.")

teste_consistencia_energia_pos_atualizacao()



## 7. Validação Preliminar contra a Solução Analítica de Onsager

Mesma ressalva dos notebooks anteriores: esta é uma checagem rápida, **não**
a validação oficial da Seção 4.4 do TCC.


In [ ]:

L_demo = 32
temperaturas_demo = np.array([1.5, 2.0, 2.269, 2.5, 3.0])

e_simulado = []
m_simulado = []
for T in temperaturas_demo:
    r = simular_swendsen_wang(L=L_demo, T=T, n_eq=50, n_prod=300)
    e_simulado.append(r["energia_por_sitio"].mean())
    m_simulado.append(r["magnetizacao_abs_por_sitio"].mean())

e_simulado = np.array(e_simulado)
m_simulado = np.array(m_simulado)

temperaturas_finas = np.linspace(1.2, 3.5, 200)
e_exato = np.array([energia_onsager(T) for T in temperaturas_finas])
m_exato = np.array([magnetizacao_onsager(T) for T in temperaturas_finas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(temperaturas_finas, e_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax1.plot(temperaturas_demo, e_simulado, "o", color="darkorange", label=f"Swendsen-Wang (L={L_demo}, demo rápida)")
ax1.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax1.set_xlabel("Temperatura $T$")
ax1.set_ylabel("Energia por sítio $e$")
ax1.legend()

ax2.plot(temperaturas_finas, m_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax2.plot(temperaturas_demo, m_simulado, "o", color="darkorange", label=f"Swendsen-Wang (L={L_demo}, demo rápida)")
ax2.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax2.set_xlabel("Temperatura $T$")
ax2.set_ylabel("Magnetização absoluta por sítio $|m|$")
ax2.legend()

plt.suptitle("Validação preliminar (demonstração rápida -- não é a validação oficial do TCC)")
plt.tight_layout()
plt.show()



## Próximos Passos

- **Validação oficial (Seção 4.4 do TCC)** e **testes de viés (4.4.3)**,
  agora com os três algoritmos implementados.
- **Notebook de autocorrelação (Seção 4.7)**: cálculo de $\tau_{\text{int}}$
  via FFT para os três algoritmos, usando as séries de energia/magnetização
  geradas pelas funções `simular_*` de cada notebook.
- **Notebook de desempenho (Seção 4.8)**: medição de $t_{\text{sweep}}$ e do
  Fator de Mérito para os três métodos, sob os mesmos $L$ e $T$.
- **Nota sobre desempenho da identificação de aglomerados**: a implementação
  de `identificar_aglomerados` aqui usada (Union-Find) tem complexidade
  quase-linear, mas o laço Python em `atualizar_aglomerados` (percorrendo
  sítio a sítio) é um candidato natural a otimização futura -- fora do
  escopo desta etapa da pesquisa (Seção 4.9 do TCC).
